In [15]:
import polars as pl
from datetime import date

pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(1000) 
pl.Config.set_tbl_width_chars(1000)

df = pl.read_csv('weather_data_us.csv')

df.head(10)


ID,DATE,TMAX,TMIN,EVAP,PRCP,Latitude,Longitude,Elevation
str,str,i64,i64,str,i64,f64,f64,f64
"""USS0012M13S""","""3/26/1997""",null,null,null,0,37.66,-112.74,2920.0
"""US1OKCV0021""","""4/18/2007""",null,null,null,241,35.1715,-97.4262,355.1
"""USC00355174""","""10/7/1999""",null,null,null,null,42.0078,-121.3186,1410.3
"""US1MDHW0012""","""10/14/2015""",null,null,null,0,39.3387,-76.9468,166.1
"""US1TXHRS014""","""10/4/2021""",null,null,null,0,32.7061,-94.1683,53.6
"""USR0000AGOP""","""3/7/2011""",-28,-200,null,null,64.2381,-145.2669,463.3
"""USC00201675""","""9/26/2012""",200,100,null,0,41.9622,-84.9925,299.9
"""USC00389039""","""12/3/2000""",33,-11,null,0,33.9,-80.5206,76.2
"""USC00230657""","""5/10/2007""",267,139,null,3,37.0539,-93.5756,399.3


In [ ]:
df = df.rename({"TMAX" : "MAX TEMP", "TMIN" : "MIN TEMP", "EVAP" : "EVAPORATION", "PRCP" : "PRECIPITATION"})

In [ ]:
df = df.rename(str.lower)

In [ ]:
df.head(10)

In [ ]:
filtered_df = df.with_columns(
    pl.col("date").str.to_date("%m/%d/%Y") # Parse format
).filter(
    pl.col("date") >= date(2010, 1, 1)     # Keep 2010 and later
)

In [ ]:
filtered_df.write_csv("2010_or_later.csv")
print("CSV file 'output.csv' created successfully.")


In [ ]:
filtered_df.head(10)

In [10]:
import polars as pl
import polars.selectors as cs


def quantDDA(df_input: pl.DataFrame) -> pl.DataFrame:
    # 1. Grab numeric columns and define the total observation count
    numeric_cols = df_input.select(cs.numeric()).columns
    total_obs = len(df_input)  # Defined as lowercase [cite: 24]
    
    # 2. Base metrics for ALL columns
    stats = [
        pl.lit(total_obs).alias("Total_Obs"),
        cs.all().count().name.prefix("Entries_"),
        cs.all().n_unique().name.prefix("Unique_"),
        cs.all().null_count().name.prefix("Null_"),
    ]

    # 3. Numeric-specific metrics
    for col in numeric_cols:
        series = df_input.get_column(col)
        # Calculate quantiles once to avoid multiple passes [cite: 35, 41]
        q1 = series.quantile(0.25)
        q2 = series.quantile(0.5)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        p01, p99 = series.quantile(0.01), series.quantile(0.99) 

        stats.extend([
            ((pl.col(col) < (q1 - 1.5 * iqr)) | (pl.col(col) > (q3 + 1.5 * iqr))).sum().alias(f"Outliers_{col}"),
            ((pl.col(col) < p01) | (pl.col(col) > p99)).sum().alias(f"Extreme_{col}"),
            pl.col(col).mode().first().alias(f"Mode_{col}"),
            pl.col(col).mean().round(2).alias(f"Mean_{col}"),
            pl.col(col).std().round(2).alias(f"StdDev_{col}"),
            pl.col(col).max().alias(f"Max_{col}"),
            pl.col(col).min().alias(f"Min_{col}"),
            pl.lit(q1).alias(f"Q1_{col}"),
            pl.lit(q2).alias(f"Q2_{col}"),
            pl.lit(q3).alias(f"Q3_{col}"),
            pl.col(col).skew().round(4).alias(f"Skew_{col}"),
            pl.col(col).kurtosis().round(4).alias(f"Kurt_{col}"),
        ])

    # 4. Add the Percent Presence metric (Fixed Variable Name) [cite: 147]
    # This uses (count / total_obs) * 100 to show the percentage of non-null values
    stats.append(
        ((cs.all().count() / total_obs) * 100).round(2).name.prefix("NonMissingPct_")
    )

    # 5. Execute aggregations and Pivot [cite: 64, 74]
    res_wide = df_input.select(stats)
    res_long = res_wide.melt().with_columns(
        pl.col("variable").str.splitn("_", 2).struct.field("field_0").alias("Stat"),
        pl.col("variable").str.splitn("_", 2).struct.field("field_1").alias("Feature")
    )

    final_df = res_long.pivot(index="Feature", on="Stat", values="value")
    
    # 6. Final Formatting [cite: 76, 79]
    final_df = final_df.fill_null("-")
    
    # Ensure NonMissingPct is the last column
    if "NonMissingPct" in final_df.columns:
        cols = [c for c in final_df.columns if c != "NonMissingPct"] + ["NonMissingPct"]
        final_df = final_df.select(cols)
    
    return final_df

In [14]:
quantDDA(filtered_df)

NameError: name 'filtered_df' is not defined

In [7]:
import polars as pl

df_test = pl.read_csv('weather_data_us.csv')

df_less_than_85 = df_test.filter(pl.col("Longitude") >= -85)

df_less_than_85.head(10)

ID,DATE,TMAX,TMIN,EVAP,PRCP,Latitude,Longitude,Elevation
str,str,i64,i64,str,i64,f64,f64,f64
"""US1MDHW0012""","""10/14/2015""",null,null,null,0,39.3387,-76.9468,166.1
"""USC00201675""","""9/26/2012""",200,100,null,0,41.9622,-84.9925,299.9
"""USC00389039""","""12/3/2000""",33,-11,null,0,33.9,-80.5206,76.2
"""USC00206510""","""8/2/2018""",267,156,null,28,45.3614,-84.9511,228.0
"""USC00445150""","""1/5/1997""",239,39,null,0,38.3683,-78.2503,175.9
"""USW00013728""","""2/8/2021""",106,-55,null,0,36.5728,-79.335,168.2
"""US1RINW0003""","""7/29/2008""",null,null,null,0,41.5328,-71.3822,36.9
"""US1PALH0005""","""7/20/2016""",null,null,null,0,40.6474,-75.6505,143.9
"""USW00013866""","""12/1/2002""",6,-44,null,3,38.3794,-81.5911,278.0


In [11]:
df_less_than_85.write_csv("less_than_85.csv")


In [12]:
quantDDA(df_less_than_85)


/var/folders/p0/bwyhv2hx34gg8rw1ds61z4k80000gn/T/ipykernel_71450/53719262.py:51: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  res_long = res_wide.melt().with_columns(


Feature,Total,Entries,Unique,Null,Outliers,Extreme,Mode,Mean,StdDev,Max,Min,Q1,Q2,Q3,Skew,Kurt,NonMissingPct
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Obs""",3.5947347e7,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ID""",null,3.5947347e7,15605.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,100.0
"""DATE""",null,3.5947347e7,10593.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,100.0
"""TMAX""",null,1.6249587e7,752.0,1.969776e7,14542.0,271201.0,null,185.7,108.81,5072.0,-728.0,106.0,206.0,278.0,-0.4705,1.8617,45.2
"""TMIN""",null,1.6239025e7,766.0,1.9708322e7,44754.0,294410.0,null,70.05,104.23,2183.0,-1022.0,-6.0,78.0,156.0,-0.3827,0.0221,45.17
"""EVAP""",null,246883.0,663.0,3.5700464e7,null,null,null,null,null,null,null,null,null,null,null,null,0.69
"""PRCP""",null,3.4280748e7,1906.0,1.666599e6,6.134319e6,340880.0,0.0,35.77,96.84,20066.0,0.0,0.0,0.0,20.0,6.1935,122.9253,95.36
"""Latitude""",null,3.5947347e7,14513.0,0.0,93082.0,714108.0,44.42,37.88,4.73,52.8333,24.5507,34.9444,38.8025,41.6333,-0.6359,-0.1972,100.0
"""Longitude""",null,3.5947347e7,14502.0,0.0,1866.0,715648.0,-81.8742,-78.87,4.61,179.2833,-85.0,-82.4,-79.8181,-75.784,9.1324,479.9747,100.0


In [16]:
quantDDA(df)

/var/folders/p0/bwyhv2hx34gg8rw1ds61z4k80000gn/T/ipykernel_71450/53719262.py:51: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  res_long = res_wide.melt().with_columns(


Feature,Total,Entries,Unique,Null,Outliers,Extreme,Mode,Mean,StdDev,Max,Min,Q1,Q2,Q3,Skew,Kurt,NonMissingPct
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Obs""",1.55840906e8,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ID""",null,1.55840906e8,56950.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,100.0
"""DATE""",null,1.55840906e8,10593.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,100.0
"""TMAX""",null,7.960931e7,2062.0,7.6231596e7,275212.0,1.478752e6,null,174.17,127.85,55372.0,-4887.0,89.0,189.0,267.0,28.1475,6791.6282,51.08
"""TMIN""",null,7.956295e7,1692.0,7.6277956e7,663257.0,1.448632e6,null,47.5,131.43,55372.0,-4116.0,-22.0,50.0,128.0,72.723,16516.6391,51.05
"""EVAP""",null,1.708841e6,1321.0,1.54132065e8,null,null,null,null,null,null,null,null,null,null,null,null,1.1
"""PRCP""",null,1.41642128e8,2433.0,1.4198778e7,2.790141e7,1.399207e6,0.0,27.29,86.47,53365.0,0.0,0.0,0.0,8.0,9.3722,1302.0159,90.89
"""Latitude""",null,1.55840906e8,47359.0,0.0,5.132201e6,3.108254e6,45.19,39.0,6.34,71.3244,19.0106,34.9628,39.3075,42.6419,0.5649,3.0157,100.0
"""Longitude""",null,1.55840906e8,52327.0,0.0,4.418158e6,3.112404e6,-106.38,-98.82,17.22,179.2833,-176.65,-109.57,-97.0656,-85.8834,-0.8447,2.0143,100.0
